<a href="https://colab.research.google.com/github/antonDinkov/AI_Agents_and_Workflows_for_Developers/blob/main/LangChain_Agents_Tools_Exercise_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai langchain-community langchain-chroma langchain-text-splitters

In [30]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_community.document_loaders import TextLoader
from google.colab import userdata
from langchain.messages import HumanMessage
from langchain_core.messages import BaseMessage
from typing import List

In [3]:
def print_conversation(messages: List[BaseMessage]) -> None:
  for message in messages:
    message.pretty_print()

In [41]:
api_key_openai = userdata.get("OPEN_AI_API_KEY")
model_instance = ChatOpenAI(
    model="gpt-5.6-luna",
    api_key=api_key_openai,
    use_responses_api=True
)

In [23]:
chroma_db = Chroma(collection_name="faq", persist_directory="/content/chroma")

In [24]:
text_loader = TextLoader(file_path="/content/FAQ.md")
md_header_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Header 1"), ("##", "Question")], strip_headers=False)

In [25]:
documents = text_loader.load()
chunk_to_embed = []
for document in documents:
  chunk_to_embed.extend(md_header_splitter.split_text(document.page_content))

In [26]:
chroma_db.add_documents(chunk_to_embed)

['a9fc029a-eb8e-4e36-8090-b60a69b357e7',
 'f04b9fbd-fca1-480e-bd8a-742fb3577e4d',
 'dbd5ba3b-dc76-46f6-abbf-43b692dcdf83',
 'dad4f4f4-2655-4339-8ae5-e7a400f12f98',
 'c9751e9e-a41a-4eed-85a2-3ff55381b4da',
 'f96e1e2c-68a2-485f-8910-2628372a24c2',
 'bebd204b-b194-4689-a588-fd8b8f97304c',
 'd7a0334e-1ce6-4a12-8ead-dd4567e0164a',
 'b1c98741-62d2-4509-9e14-b1a8e9836042',
 '21afdeea-bed5-4d5d-856d-60f115b5c9d3',
 '427edec1-068c-40a9-9f42-641544ea38e5',
 '497644e4-ef30-45a3-94a1-815df97a1324',
 '8b45ba6c-d75c-499d-ae90-ef3f2e451ff1',
 'aae47be5-d003-46a4-a1a2-d7553d53842d',
 '349b9017-276b-4e1e-a3a2-30efa9fad72b',
 'be00ba69-db99-4623-a12f-8d4b8f9f4a63',
 '047b6043-fe62-4081-8ace-abef436449ea']

In [37]:
@tool
def search_database (query: str) -> str:
  """
  Call this tool to search the internal database using a natural language query.
  """
  results = chroma_db.similarity_search(query, k=3)
  return "\n\n------\n\n".join(d.page_content for d in results)

In [38]:
print(search_database.invoke("delivery"))

## Q14. Do you deliver on weekends and public holidays?  
Standard processing and delivery times are based on **business days**.  
Orders placed during weekends or public holidays are generally processed on the next business day.  
Weekend delivery may depend on courier availability and the destination.  
---

------

## Q14. Do you deliver on weekends and public holidays?  
Standard processing and delivery times are based on **business days**.  
Orders placed during weekends or public holidays are generally processed on the next business day.  
Weekend delivery may depend on courier availability and the destination.  
---

------

## Q5. What are the delivery times?  
### Sofia  
Orders placed before **4:00 PM on a business day** are normally delivered within **24 hours**.  
Orders placed after 4:00 PM, during weekends, or on public holidays are processed on the next business day.  
### Nationwide delivery  
Orders to other locations in Bulgaria are normally delivered within **1–3 bus

In [43]:
agent = create_agent(
    model=model_instance,
    tools=[
      search_database
    ],
    system_prompt="You are a helpful customer support agent",
    debug=True
)

In [44]:
answer = agent.invoke(input={"messages": [HumanMessage("Hello! How long should I wait for a standard delivery? Do you offer an express option?")]})

[values] {'messages': [HumanMessage(content='Hello! How long should I wait for a standard delivery? Do you offer an express option?', additional_kwargs={}, response_metadata={}, id='2191b872-5f06-4e11-a6b5-181450360478')]}
[updates] {'model': {'messages': [AIMessage(content=[{'id': 'rs_098f3bceeba3f49c006a8c6383ade487d2ac66f4b0dd015c46', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqjGOEQ7emRQfK3mUAn0GiFDfKzUjOe1SIpybWyf9O2Gkj58M-6zwqTrCr9mQ-qTRYQnLyTOCF8YVLXa0uKaT3Q4o6zRUPnWJvohQ1O_zg0t3Bbowou_gze-RvKoxpTod0eUvO2_Jlj2iSYdfsJMdJe54mHHT4M_5uqtPq9Qd2vj1Ye40MCwIDOJM-IX-OAIlsq91g2xsYPwhrvy37tocT4a6gtg4hNV4Mj0fyAPe7kZqyWgJ75s0ZIKVFRn9W85RDV1NYn7PMlZmVNg8iQFZ4uqikTS3qkZ02LhVZYs-D1goe0okKdzBmJRMDcZw-ZtRjGrEF95jMwtSttkpuQ17PVqMK64b3pHOrOKZBeAxXA-o1dp1Kqu0CVMo5UO2speJySJosoyWp8yZvTrkWtCM-WaI_LsvsztoOTIt-LFR8L1MnuwLXZMH-FVfVmsW-uF1hhQNUxkFJOrpszWBKRSNoElm_ZVX-0PTiW3vluJG0gXSdqrt-Mz2_YdBftcJA9a8uxY0uZ_bIWf3O2dO9yBE_ZWCusYtwggN0KjlArqlcFoJHQEDnqhgZU1O8EbR7JoLKrv9

In [45]:
print_conversation(answer['messages'])

================================ Human Message =================================

Hello! How long should I wait for a standard delivery? Do you offer an express option?
================================== Ai Message ==================================

[{'id': 'rs_098f3bceeba3f49c006a8c6383ade487d2ac66f4b0dd015c46', 'summary': [], 'type': 'reasoning', 'content': [], 'encrypted_content': 'gAAAAABqjGOEQ7emRQfK3mUAn0GiFDfKzUjOe1SIpybWyf9O2Gkj58M-6zwqTrCr9mQ-qTRYQnLyTOCF8YVLXa0uKaT3Q4o6zRUPnWJvohQ1O_zg0t3Bbowou_gze-RvKoxpTod0eUvO2_Jlj2iSYdfsJMdJe54mHHT4M_5uqtPq9Qd2vj1Ye40MCwIDOJM-IX-OAIlsq91g2xsYPwhrvy37tocT4a6gtg4hNV4Mj0fyAPe7kZqyWgJ75s0ZIKVFRn9W85RDV1NYn7PMlZmVNg8iQFZ4uqikTS3qkZ02LhVZYs-D1goe0okKdzBmJRMDcZw-ZtRjGrEF95jMwtSttkpuQ17PVqMK64b3pHOrOKZBeAxXA-o1dp1Kqu0CVMo5UO2speJySJosoyWp8yZvTrkWtCM-WaI_LsvsztoOTIt-LFR8L1MnuwLXZMH-FVfVmsW-uF1hhQNUxkFJOrpszWBKRSNoElm_ZVX-0PTiW3vluJG0gXSdqrt-Mz2_YdBftcJA9a8uxY0uZ_bIWf3O2dO9yBE_ZWCusYtwggN0KjlArqlcFoJHQEDnqhgZU1O8EbR7JoLKrv9t7oMKQoFnsbTD5BjIWCD3RQ7